In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load cleaned dataset
movies = pd.read_csv('movies_cleaned.csv')

print(f"Shape  : {movies.shape}")
print(f"Columns: {movies.columns.tolist()}")
print(f"\nSample tags:")
print(f"🎬 {movies['title'][0]}")
print(f"   {movies['tags'][0][:200]}")

Shape  : (3146, 15)
Columns: ['id', 'title', 'overview', 'tagline', 'genres', 'keywords', 'cast', 'director', 'vote_average', 'vote_count', 'popularity', 'release_year', 'runtime', 'original_language', 'tags']

Sample tags:
🎬 avatar
   22nd century paraplegic marine dispatched moon pandora unique mission becomes torn following orders protecting alien civilization enter world pandora action adventure fantasy sciencefiction culturecla


Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,   
    ngram_range=(1, 2),  
    min_df=2             
)

# Fit and transform tags
tfidf_matrix = tfidf.fit_transform(movies['tags'])

print(f"TF-IDF matrix shape : {tfidf_matrix.shape}")
print(f"  rows = movies      : {tfidf_matrix.shape[0]}")
print(f"  cols = features    : {tfidf_matrix.shape[1]}")
print(f"\nSample feature names:")
print(tfidf.get_feature_names_out()[:20])
print(f"\nMatrix density      : {tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]):.4f}")

TF-IDF matrix shape : (3146, 5000)
  rows = movies      : 3146
  cols = features    : 5000

Sample feature names:
['007' '10' '10 years' '100' '10yearold' '11' '12' '12yearold' '13' '15'
 '15yearold' '17yearold' '1930s' '1940s' '1950s' '1960s' '1970s' '1980s'
 '19th' '19th century']

Matrix density      : 0.0072


In [4]:
# Compute cosine similarity between all movies
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"Cosine similarity matrix shape : {cosine_sim.shape}")
print(f"Memory usage                   : {cosine_sim.nbytes / 1024**2:.2f} MB")
print(f"\nSample similarity scores (Avatar vs first 5 movies):")
for i, score in enumerate(cosine_sim[0][:5]):
    print(f"  {movies['title'][i]:40s} → {score:.4f}")

print(f"\nMax similarity (excluding self) : {sorted(cosine_sim[0], reverse=True)[1]:.4f}")
print(f"Min similarity                  : {cosine_sim[0].min():.4f}")
print(f"Mean similarity                 : {cosine_sim[0].mean():.4f}")

Cosine similarity matrix shape : (3146, 3146)
Memory usage                   : 75.51 MB

Sample similarity scores (Avatar vs first 5 movies):
  avatar                                   → 1.0000
  pirates of the caribbean at worlds end   → 0.0401
  spectre                                  → 0.0192
  the dark knight rises                    → 0.0196
  john carter                              → 0.1146

Max similarity (excluding self) : 0.1793
Min similarity                  : 0.0000
Mean similarity                 : 0.0163


In [5]:
# Create title-to-index mapping
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

def recommend(title, n=10):
    if title not in indices:
        print(f"❌ '{title}' not found!")
        matches = [t for t in movies['title'] if title.split()[0] in t]
        print("Did you mean:")
        for m in matches[:5]:
            print(f"   → {m}")
        return

    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+1]

    movie_indices = [i[0] for i in sim_scores]
    similarity    = [round(i[1], 4) for i in sim_scores]

    results = movies[['title', 'genres', 'vote_average',
                       'release_year', 'director']].iloc[movie_indices].copy()
    results['similarity'] = similarity
    results = results.reset_index(drop=True)
    results.index += 1

    print(f"\n🎬 Because you watched: '{title}'")
    print(f"{'='*65}")
    print(f"{'#':<4} {'Title':<35} {'Sim':>6} {'Rating':>7} {'Year':>6}")
    print(f"{'-'*65}")
    for i, row in results.iterrows():
        print(f"{i:<4} {row['title'][:34]:<35} {row['similarity']:>6.4f} "
              f"{row['vote_average']:>7.1f} {int(row['release_year']):>6}")
    return results

In [6]:
print("\n" + "="*65)
r1 = recommend('avatar')

print("\n" + "="*65)
r2 = recommend('the dark knight rises')

print("\n" + "="*65)
r3 = recommend('the godfather')



🎬 Because you watched: 'avatar'
#    Title                                  Sim  Rating   Year
-----------------------------------------------------------------
1    star trek into darkness             0.1793     7.4   2013
2    battle los angeles                  0.1767     5.5   2011
3    aliens                              0.1662     7.7   1986
4    titan ae                            0.1581     6.3   2000
5    alien                               0.1490     7.9   1979
6    apollo 18                           0.1424     5.0   2011
7    bloodrayne                          0.1416     3.5   2005
8    mad max beyond thunderdome          0.1380     5.9   1985
9    treasure planet                     0.1353     7.2   2002
10   predators                           0.1337     6.0   2010


🎬 Because you watched: 'the dark knight rises'
#    Title                                  Sim  Rating   Year
-----------------------------------------------------------------
1    the dark knight         

In [7]:
def recommend(title, n=10, min_rating=5.8):
    if title not in indices:
        print(f"❌ '{title}' not found!")
        matches = [t for t in movies['title'] if title.split()[0] in t]
        print("Did you mean:")
        for m in matches[:5]:
            print(f"   → {m}")
        return

    idx = indices[title]

    # Get similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:]  # exclude self

    # Get movie indices and build result
    movie_indices = [i[0] for i in sim_scores]
    similarity    = [round(i[1], 4) for i in sim_scores]

    results = movies[['title', 'genres', 'vote_average',
                       'vote_count', 'release_year',
                       'director', 'popularity']].iloc[movie_indices].copy()
    results['similarity'] = similarity

    # ── Filter 1: minimum rating threshold ──
    results = results[results['vote_average'] >= min_rating]

    # ── Filter 2: weighted score (similarity + rating boost) ──
    results['score'] = (results['similarity'] * 0.7 +
                       (results['vote_average'] / 10) * 0.3)

    # Take top n
    results = results.head(n).reset_index(drop=True)

In [8]:
print("\n" + "="*70)
r1 = recommend('avatar')

print("\n" + "="*70)
r2 = recommend('the dark knight rises')

print("\n" + "="*70)
r3 = recommend('the godfather')

print("\n" + "="*70)
r4 = recommend('toy story')

In [9]:
# ── Debug Cell ──

# 1. Check exact title format
print("── Searching for titles ──")
for search in ['avatar', 'godfather', 'toy story', 'dark knight']:
    matches = [t for t in movies['title'] if search in t]
    print(f"\n'{search}' matches:")
    for m in matches[:5]:
        print(f"   → '{m}'")

# 2. Check vote_average distribution
print("\n── Vote average distribution ──")
print(movies['vote_average'].describe())
print(f"\nMovies with rating >= 5.8 : {(movies['vote_average'] >= 5.8).sum()}")
print(f"Movies with rating >= 6.0 : {(movies['vote_average'] >= 6.0).sum()}")

# 3. Test raw similarity for avatar
print("\n── Raw similarity check for avatar ──")
if 'avatar' in indices:
    idx = indices['avatar']
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:6]
    for i, score in sim_scores:
        print(f"  {movies['title'][i]:40s} → sim:{score:.4f} rating:{movies['vote_average'][i]}")
else:
    print("'avatar' NOT found in indices!")
    print("Sample index keys:")
    print(list(indices.index[:10]))

── Searching for titles ──

'avatar' matches:
   → 'avatar'

'godfather' matches:
   → 'the godfather part iii'
   → 'the godfather part ii'
   → 'the godfather'

'toy story' matches:
   → 'toy story 3'
   → 'toy story 2'
   → 'toy story'

'dark knight' matches:
   → 'the dark knight rises'
   → 'the dark knight'
   → 'batman the dark knight returns part 2'

── Vote average distribution ──
count    3146.000000
mean        6.356675
std         0.834957
min         2.900000
25%         5.800000
50%         6.400000
75%         7.000000
max         8.500000
Name: vote_average, dtype: float64

Movies with rating >= 5.8 : 2436
Movies with rating >= 6.0 : 2179

── Raw similarity check for avatar ──
  star trek into darkness                  → sim:0.1793 rating:7.4
  battle los angeles                       → sim:0.1767 rating:5.5
  aliens                                   → sim:0.1662 rating:7.7
  titan ae                                 → sim:0.1581 rating:6.3
  alien                       

In [10]:
def recommend(title, n=10, min_rating=5.5):
    if title not in indices:
        print(f"❌ '{title}' not found!")
        matches = [t for t in movies['title'] if title.split()[0] in t]
        print("Did you mean:")
        for m in matches[:5]:
            print(f"   → {m}")
        return

    idx = indices[title]

    # Get similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:]  # exclude self

    # Get ALL indices first
    movie_indices = [i[0] for i in sim_scores]
    similarity    = [round(i[1], 4) for i in sim_scores]

    # Build full results dataframe
    results = movies[['title', 'genres', 'vote_average',
                       'vote_count', 'release_year',
                       'director']].iloc[movie_indices].copy()
    results['similarity'] = similarity

    # ── Filter: minimum rating ──
    results = results[results['vote_average'] >= min_rating].copy()

    # ── Weighted score ──
    results['score'] = ((results['similarity'] * 0.7) +
                        ((results['vote_average'] / 10) * 0.3)).round(4)

    # ── Sort by score descending ──
    results = results.sort_values('score', ascending=False)

    # Take top n
    results = results.head(n).reset_index(drop=True)
    results.index += 1

    print(f"\n🎬 Because you watched: '{title}'")
    print(f"{'='*72}")
    print(f"{'#':<4} {'Title':<35} {'Score':>6} {'Sim':>6} {'Rating':>7} {'Year':>6}")
    print(f"{'-'*72}")
    for i, row in results.iterrows():
        print(f"{i:<4} {row['title'][:34]:<35} {row['score']:>6.4f} "
              f"{row['similarity']:>6.4f} {row['vote_average']:>7.1f} "
              f"{int(row['release_year']):>6}")
    return results

# ── Test ──
print("\n" + "="*72)
r1 = recommend('avatar')

print("\n" + "="*72)
r2 = recommend('the dark knight rises')

print("\n" + "="*72)
r3 = recommend('the godfather')

print("\n" + "="*72)
r4 = recommend('toy story')



🎬 Because you watched: 'avatar'
#    Title                                Score    Sim  Rating   Year
------------------------------------------------------------------------
1    star trek into darkness             0.3475 0.1793     7.4   2013
2    aliens                              0.3473 0.1662     7.7   1986
3    alien                               0.3413 0.1490     7.9   1979
4    treasure planet                     0.3107 0.1353     7.2   2002
5    the book of life                    0.3096 0.1295     7.3   2014
6    edge of tomorrow                    0.3044 0.1091     7.6   2014
7    the fifth element                   0.3026 0.1195     7.3   1997
8    titan ae                            0.2997 0.1581     6.3   2000
9    the thing                           0.2980 0.0915     7.8   1982
10   xmen days of future past            0.2961 0.1016     7.5   2014


🎬 Because you watched: 'the dark knight rises'
#    Title                                Score    Sim  Rating   Year
----

In [11]:
# Paste full output for godfather and toy story here
print("\n" + "="*72)
r3 = recommend('the godfather')

print("\n" + "="*72)
r4 = recommend('toy story')



🎬 Because you watched: 'the godfather'
#    Title                                Score    Sim  Rating   Year
------------------------------------------------------------------------
1    the godfather part ii               0.4549 0.2942     8.3   1974
2    the godfather part iii              0.3626 0.2137     7.1   1990
3    goodfellas                          0.3378 0.1311     8.2   1990
4    the shawshank redemption            0.3211 0.0944     8.5   1994
5    donnie brasco                       0.3128 0.1297     7.4   1997
6    apocalypse now                      0.2965 0.0807     8.0   1979
7    once upon a time in america         0.2960 0.0714     8.2   1984
8    casino                              0.2900 0.0800     7.8   1995
9    the usual suspects                  0.2891 0.0659     8.1   1995
10   muppets most wanted                 0.2887 0.1467     6.2   2014


🎬 Because you watched: 'toy story'
#    Title                                Score    Sim  Rating   Year
---------

In [12]:
def recommend(title, n=10, min_rating=5.5, min_similarity=0.08):
    if title not in indices:
        print(f"❌ '{title}' not found!")
        matches = [t for t in movies['title'] if title.split()[0] in t]
        print("Did you mean:")
        for m in matches[:5]:
            print(f"   → {m}")
        return

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:]

    movie_indices = [i[0] for i in sim_scores]
    similarity    = [round(i[1], 4) for i in sim_scores]

    results = movies[['title', 'genres', 'vote_average',
                       'vote_count', 'release_year',
                       'director']].iloc[movie_indices].copy()
    results['similarity'] = similarity

    # ── Filter 1: minimum rating ──
    results = results[results['vote_average'] >= min_rating].copy()

    # ── Filter 2: minimum similarity threshold ──  ← NEW
    results = results[results['similarity'] >= min_similarity].copy()

    # ── Weighted score ──
    results['score'] = ((results['similarity'] * 0.7) +
                        ((results['vote_average'] / 10) * 0.3)).round(4)

    # ── Sort by score ──
    results = results.sort_values('score', ascending=False)

    results = results.head(n).reset_index(drop=True)
    results.index += 1

    print(f"\n🎬 Because you watched: '{title}'")
    print(f"{'='*72}")
    print(f"{'#':<4} {'Title':<35} {'Score':>6} {'Sim':>6} {'Rating':>7} {'Year':>6}")
    print(f"{'-'*72}")
    for i, row in results.iterrows():
        print(f"{i:<4} {row['title'][:34]:<35} {row['score']:>6.4f} "
              f"{row['similarity']:>6.4f} {row['vote_average']:>7.1f} "
              f"{int(row['release_year']):>6}")

    # ── Warn if fewer than n results ──
    if len(results) < n:
        print(f"\n⚠️  Only {len(results)} results found above thresholds")
        print(f"   Try lowering min_similarity={min_similarity} or min_rating={min_rating}")

    return results

# ── Test all 4 ──
for title in ['avatar', 'the dark knight rises', 'the godfather', 'toy story']:
    print("\n" + "="*72)
    recommend(title)



🎬 Because you watched: 'avatar'
#    Title                                Score    Sim  Rating   Year
------------------------------------------------------------------------
1    star trek into darkness             0.3475 0.1793     7.4   2013
2    aliens                              0.3473 0.1662     7.7   1986
3    alien                               0.3413 0.1490     7.9   1979
4    treasure planet                     0.3107 0.1353     7.2   2002
5    the book of life                    0.3096 0.1295     7.3   2014
6    edge of tomorrow                    0.3044 0.1091     7.6   2014
7    the fifth element                   0.3026 0.1195     7.3   1997
8    titan ae                            0.2997 0.1581     6.3   2000
9    the thing                           0.2980 0.0915     7.8   1982
10   xmen days of future past            0.2961 0.1016     7.5   2014


🎬 Because you watched: 'the dark knight rises'
#    Title                                Score    Sim  Rating   Year
----

In [13]:
# ── Rebuild tags with boosted important features ──

def build_boosted_tags(row):
    # Repeat director 3x and genres 2x to increase their weight
    return (
        row['overview_tokens']          +   # plot
        row['tagline_tokens']           +   # tagline
        row['genres']          * 2      +   # genres × 2
        row['keywords']                 +   # keywords
        row['cast']                     +   # cast
        row['director']        * 3          # director × 3 ← most important
    )

# Check if overview_tokens and tagline_tokens still exist
print("── Columns available ──")
print(movies.columns.tolist())
print(f"\nSample genres : {movies['genres'][0]}")
print(f"Sample director: {movies['director'][0]}")

── Columns available ──
['id', 'title', 'overview', 'tagline', 'genres', 'keywords', 'cast', 'director', 'vote_average', 'vote_count', 'popularity', 'release_year', 'runtime', 'original_language', 'tags']

Sample genres : ['action', 'adventure', 'fantasy', 'sciencefiction']
Sample director: ['jamescameron']


In [15]:
import ast

# ── Fix 1: tagline NaN → empty string ──
movies['tagline'] = movies['tagline'].fillna('')

# ── Fix 2: Parse list columns from string back to lists ──
for col in ['genres', 'keywords', 'cast', 'director']:
    movies[col] = movies[col].apply(lambda x: ast.literal_eval(x) 
                                    if isinstance(x, str) else x)

# ── Fix 3: Rebuild tokens ──
movies['overview_tokens'] = movies['overview'].apply(lambda x: x.split() if isinstance(x, str) else [])
movies['tagline_tokens']  = movies['tagline'].apply(lambda x: x.split() if isinstance(x, str) and x else [])

# ── Verify ──
print(f"Null taglines  : {movies['tagline'].isnull().sum()}")
print(f"Null overviews : {movies['overview'].isnull().sum()}")
print(f"Sample genres  : {movies['genres'][0]}")
print(f"Sample director: {movies['director'][0]}")
print(f"Sample tagline tokens: {movies['tagline_tokens'][0]}")
print(f"Sample overview tokens: {movies['overview_tokens'][0][:5]}")

Null taglines  : 0
Null overviews : 0
Sample genres  : ['action', 'adventure', 'fantasy', 'sciencefiction']
Sample director: ['jamescameron']
Sample tagline tokens: ['enter', 'the', 'world', 'of', 'pandora']
Sample overview tokens: ['in', 'the', '22nd', 'century', 'a']


In [19]:
# Install nltk
import subprocess
subprocess.run(['pip', 'install', 'nltk'], capture_output=True)

import nltk
nltk.download('stopwords')

print("✅ NLTK installed and stopwords downloaded!")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


✅ NLTK installed and stopwords downloaded!


In [20]:
from nltk.corpus import stopwords

def build_boosted_tags(row):
    return (
        row['overview_tokens']     +   # plot
        row['tagline_tokens']      +   # tagline
        row['genres']        * 2   +   # genres × 2
        row['keywords']            +   # keywords
        row['cast']                +   # cast
        row['director']      * 3       # director × 3
    )

movies['tags'] = movies.apply(build_boosted_tags, axis=1)
movies['tags'] = movies['tags'].apply(lambda x: ' '.join(x))

# Remove stop words & noise
stop_words  = set(stopwords.words('english'))
noise_words = ['duringcreditsstinger', 'aftercreditsstinger']

movies['tags'] = movies['tags'].apply(
    lambda x: ' '.join([w for w in x.split() 
                        if w not in stop_words 
                        and w not in noise_words])
)

movies.drop(columns=['overview_tokens', 'tagline_tokens'], 
            inplace=True, errors='ignore')

print(f"✅ Tags rebuilt!")
print(f"\n🎬 {movies['title'][0]}")
print(f"   {movies['tags'][0][:300]}")
print(f"\nDirector appears : {movies['tags'][0].count('jamescameron')}x")
print(f"Genre appears    : {movies['tags'][0].count('action')}x")

✅ Tags rebuilt!

🎬 avatar
   22nd century paraplegic marine dispatched moon pandora unique mission becomes torn following orders protecting alien civilization enter world pandora action adventure fantasy sciencefiction action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristi

Director appears : 3x
Genre appears    : 2x


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)

tfidf_matrix = tfidf.fit_transform(movies['tags'])
cosine_sim   = cosine_similarity(tfidf_matrix, tfidf_matrix)
indices      = pd.Series(movies.index, 
                         index=movies['title']).drop_duplicates()

print(f"TF-IDF matrix : {tfidf_matrix.shape}")
print(f"Cosine sim    : {cosine_sim.shape}")
print(f"Indices       : {len(indices)}")

TF-IDF matrix : (3146, 5000)
Cosine sim    : (3146, 3146)
Indices       : 3146


In [22]:
for title in ['avatar', 'the dark knight rises', 
              'the godfather', 'toy story']:
    print("\n" + "="*72)
    recommend(title)



🎬 Because you watched: 'avatar'
#    Title                                Score    Sim  Rating   Year
------------------------------------------------------------------------
1    aliens                              0.5167 0.4082     7.7   1986
2    terminator 2 judgment day           0.4668 0.3368     7.7   1991
3    the terminator                      0.4646 0.3508     7.3   1984
4    the abyss                           0.4470 0.3343     7.1   1989
5    true lies                           0.4379 0.3341     6.8   1994
6    titanic                             0.4066 0.2595     7.5   1997
7    xmen days of future past            0.3671 0.2030     7.5   2014
8    small soldiers                      0.3368 0.2154     6.2   1998
9    superman                            0.3273 0.1718     6.9   1978
10   man of steel                        0.3252 0.1860     6.5   2013


🎬 Because you watched: 'the dark knight rises'
#    Title                                Score    Sim  Rating   Year
----

In [23]:
# Show godfather and toy story full output
print("\n" + "="*72)
recommend('the godfather')

print("\n" + "="*72)
recommend('toy story')



🎬 Because you watched: 'the godfather'
#    Title                                Score    Sim  Rating   Year
------------------------------------------------------------------------
1    the godfather part ii               0.6223 0.5333     8.3   1974
2    the godfather part iii              0.5208 0.4397     7.1   1990
3    the rainmaker                       0.4665 0.3793     6.7   1997
4    the outsiders                       0.4577 0.3581     6.9   1983
5    apocalypse now                      0.4466 0.2952     8.0   1979
6    the conversation                    0.4403 0.3076     7.5   1974
7    dracula                             0.4267 0.3053     7.1   1992
8    peggy sue got married               0.4222 0.3503     5.9   1986
9    goodfellas                          0.3534 0.1535     8.2   1990
10   the shawshank redemption            0.3383 0.1190     8.5   1994


🎬 Because you watched: 'toy story'
#    Title                                Score    Sim  Rating   Year
---------

,title,genres,vote_average,vote_count,release_year,director,similarity,score
1,toy story 2,"[animation, comedy, family]",7.3,3806,1999,[johnlasseter],0.5981,0.6377
2,a bugs life,"[adventure, animation, comedy, family]",6.8,2303,1998,"[andrewstanton, johnlasseter]",0.3318,0.4363
3,toy story 3,"[animation, family, comedy]",7.6,4597,2010,[leeunkrich],0.2766,0.4216
4,cars,"[animation, adventure, comedy, family]",6.6,3877,2006,"[johnlasseter, joeranft]",0.3080,0.4136
5,cars 2,"[animation, family, adventure, comedy]",5.8,2033,2011,"[johnlasseter, bradlewis]",0.2912,0.3778
6,the simpsons movie,"[animation, comedy, family]",6.9,2264,2007,[davidsilverman],0.2063,0.3514
7,monsters inc,"[animation, comedy, family]",7.5,5996,2001,[petedocter],0.1743,0.3470
8,the boxtrolls,"[animation, comedy, family, fantasy]",6.6,668,2014,"[anthonystacchi, grahamannable]",0.2061,0.3423
9,the emperors new groove,"[adventure, animation, comedy, family, fantasy]",7.2,1490,2000,[markdindal],0.1751,0.3386
10,up,"[animation, comedy, family, adventure]",7.7,6870,2009,[petedocter],0.1459,0.3331
